# 04 — DPO with Self-Generated Reasoning Preferences (Condition B)

**Notebook 4 of 7 (seed-parameterized — run 3 times).** Trains DPO on the 500 self-generated reasoning preference pairs produced by Notebook 02, then evaluates on GSM8K[:500].

This is the **treatment** condition. The comparison against Notebook 03 (general-purpose UltraFeedback preferences) is the central test of the main hypothesis H2 in the RESEARCH_PLAN: does a *reasoning-targeted* preference signal transfer to GSM8K where a general signal did not?

- **Prerequisite:** Notebook 02 has run, so `data/self_generated_prefs.jsonl` exists on Drive.
- **Runtime:** ~30 min train + ~1.5h eval ≈ **2 hours per seed** on Colab T4.
- **Total across 3 seeds:** ~6 hours.
- **Outputs:** `results/dpo_B_seed{1,2,3}.json` on Drive.

## 1. Install dependencies

Same stack as Notebooks 01 and 02. If you've already run those in this Colab session, the install below is a no-op.

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Configuration — **set `SEED` for this run**

You will run this notebook three times, changing `SEED` to 1, then 2, then 3. Each run produces `results/dpo_B_seed{SEED}.json` on Drive. The three seeded runs together give the sample size we need for the aggregated statistics in Notebook 05.

In [ ]:
SEED = 1   # <-- change to 2 for the second run, 3 for the third

# Sanity: fail loudly on typos
assert SEED in (1, 2, 3), f"Expected SEED in (1,2,3), got {SEED}"

## 3. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/dpo-reasoning-transfer'
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/adapters', exist_ok=True)

RESULTS_PATH  = f'{PROJECT_ROOT}/results/dpo_B_seed{SEED}.json'
ADAPTER_PATH  = f'{PROJECT_ROOT}/adapters/dpo_B_seed{SEED}'
BASELINE_PATH = f'{PROJECT_ROOT}/results/baseline_gsm8k_500.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'This run will write:')
print(f'  results: {RESULTS_PATH}')
print(f'  adapter: {ADAPTER_PATH}   # loaded by Notebook 06')

## 4. Load `src/`

In [ ]:
import sys
CODE_DIR = PROJECT_ROOT   # or f'{PROJECT_ROOT}/repo' if cloned from GitHub
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.train import attach_lora, train_dpo_and_evaluate, compare_to_baseline
print('src/train.py loaded')

## 5. Load the base model and attach LoRA adapters

Same base model as Notebooks 01 and 02. LoRA config (r=16, α=16, standard attention + MLP targets) is identical between Conditions A and B — the *only* variable between them is the preference dataset.

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

model = attach_lora(model, r=16, lora_alpha=16, seed=SEED)
print(f'Model + LoRA attached (LoRA seed = {SEED})')

## 6. Load and format the reasoning preferences

Read the JSONL Notebook 02 produced, shuffle by `SEED` for training-order variance, then apply the chat template via `format_pair_for_dpo`. Unlike Condition A (where each seed samples different 500 pairs from a very large pool), Condition B has a fixed 500-pair set that all seeds see — so the seed here controls training-order, minibatch composition, and initialization, but not the dataset content.

In [ ]:
import random
from datasets import Dataset
from src.prefs import load_preference_jsonl, format_pair_for_dpo

PREF_PATH = f'{PROJECT_ROOT}/data/self_generated_prefs.jsonl'
raw_pairs = load_preference_jsonl(PREF_PATH)
print(f'Loaded {len(raw_pairs)} raw pairs from {PREF_PATH}')

# Shuffle by SEED for training-order variance
rng = random.Random(SEED)
rng.shuffle(raw_pairs)

# Take the first 500 (they're all we have, but be explicit)
raw_pairs = raw_pairs[:500]
assert len(raw_pairs) == 500, (
    f'Only {len(raw_pairs)} pairs available; Notebook 02 needs to produce more.'
)

# Format and wrap in a datasets.Dataset
formatted_records = [format_pair_for_dpo(p, tokenizer) for p in raw_pairs]
dpo_dataset = Dataset.from_list(formatted_records)
print(f'Formatted {len(dpo_dataset)} pairs (columns: {dpo_dataset.column_names})')
print()
print('--- Example prompt (first 300 chars) ---')
print(dpo_dataset[0]['prompt'][:300])
print()
print('--- Example chosen  (first 200 chars) ---')
print(dpo_dataset[0]['chosen'][:200])
print()
print('--- Example rejected (first 200 chars) ---')
print(dpo_dataset[0]['rejected'][:200])

## 7. Train + evaluate

Same single call as Notebook 03. Skips work if `RESULTS_PATH` already exists for this (condition, seed) tuple; resumable across Colab crashes during eval.

In [ ]:
results = train_dpo_and_evaluate(
    model=model,
    tokenizer=tokenizer,
    dpo_dataset=dpo_dataset,
    results_output_path=RESULTS_PATH,
    condition_name='dpo_B_reasoning',
    preference_dataset_name=f'self_generated_prefs.jsonl[shuffle(SEED)][:500]',
    seed=SEED,
    n_eval_samples=500,
    output_dir=f'/tmp/dpo_B_seed{SEED}',
    adapter_save_path=ADAPTER_PATH,
)
print()
print('=' * 60)
print(f"Condition B, seed {SEED}: {results['accuracy']:.1%} "
      f"({results['n_correct']}/{results['n_samples']})")
print('=' * 60)

## 8. Quick comparison to baseline

A fast preview of the delta vs. baseline. **This is not the final analysis** — Notebook 05 pools across all three seeds and both conditions with proper multiple-comparison correction. But the single-seed McNemar p here is a useful reality check that the training didn't obviously break anything.

In [ ]:
import json

with open(BASELINE_PATH) as f:
    baseline_results = json.load(f)

cmp = compare_to_baseline(baseline_results, results)

print(f'Baseline accuracy:  {cmp["baseline_accuracy"]:.1%}')
print(f'Condition B (seed {SEED}):     {cmp["dpo_accuracy"]:.1%}')
print(f'Delta:              {cmp["delta_pp"]:+.1f} pp')
print()
print(f'Discordant pairs (out of {cmp["n_compared"]}):')
print(f'  baseline right, DPO wrong: {cmp["baseline_right_dpo_wrong"]}')
print(f'  baseline wrong, DPO right: {cmp["baseline_wrong_dpo_right"]}')
print()
print(f'Single-seed McNemar p (exact, two-tailed): {cmp["mcnemar_p"]:.4f}')
print()
print('Reminder: a single seed is underpowered; Notebook 05 aggregates seeds 1–3.')

## 9. Training-loss trace

Sanity check on the training itself — did DPO loss actually decrease, and did the reward margin (chosen − rejected implicit reward) go positive?

In [ ]:
import matplotlib.pyplot as plt

log_history = results['run_metadata']['training']['log_history']
losses  = [(e['step'], e['loss'])              for e in log_history if 'loss' in e]
margins = [(e['step'], e['rewards/margins'])   for e in log_history if 'rewards/margins' in e]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

if losses:
    xs, ys = zip(*losses)
    axes[0].plot(xs, ys, marker='.', color='#4C72B0')
    axes[0].set_xlabel('Training step')
    axes[0].set_ylabel('DPO loss')
    axes[0].set_title(f'Training loss (seed {SEED})')
    axes[0].grid(alpha=0.3)

if margins:
    xs, ys = zip(*margins)
    axes[1].plot(xs, ys, marker='.', color='#DD8452')
    axes[1].axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    axes[1].set_xlabel('Training step')
    axes[1].set_ylabel('rewards/margins (chosen − rejected)')
    axes[1].set_title(f'Implicit reward margin (seed {SEED})')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Next

- If this was seed 1: change `SEED` to 2 at the top and re-run cells 5 onward.
- After all three B seeds are cached alongside all three A seeds and the baseline: **Notebook 05** aggregates everything, does McNemar / bootstrap / Cochran's Q properly, and produces the main results figure for the report.